<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Noma99-ai/Excercise_DLL_Course/blob/main/exercises/Ex10.1-spm-and-thermal/Ex10.1_04_control_panel.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the set's library files from the public course repository, so nothing needs uploading.*

<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Liu, *PINN with Python*, 2025.
- Prince, *Understanding Deep Learning*, MIT Press 2023.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

**Run-ready course copy.** The implementation cells in this notebook are already filled so students can run the notebook top-to-bottom and inspect the outputs. The original exercise prompts and `TODO` comments are retained as learning cues; report/reflection text is still for the student to complete.


# Ex_10.1 · Notebook 04 — the control panel

**Paired with L10.1 · Battery models**

Vary C-rate, ambient temperature, initial SOC and cooling with sliders. The
readout shows the derived quantities that decide whether your modelling choices
are sound — current, discharge duration, particle time constant, Biot number,
and the sampling ratio, collocation points per trainable parameter.

Paste your `residual_fn` and `loss_fn_factory` from notebook 01 below.

---

## 0 · Setup

In [ ]:
# files-cell v1 ----------------------------------------------------------
# This set's library files must sit beside the notebook. Locally they
# already do. On Google Colab, where a notebook opens on its own, they are
# fetched from the public course repository. Run this cell first.
import os, urllib.request
FILES = ['course_core.py', 'pinn_core.py', 'problem.py']
URL = "https://raw.githubusercontent.com/Noma99-ai/Excercise_DLL_Course/main/exercises/Ex10.1-spm-and-thermal/"
for f in FILES:
    if not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
print("files ready:", ", ".join(FILES))


In [ ]:
# outputs-cell v1 --------------------------------------------------------
# Later notebooks in this set read results that earlier ones save. On Google
# Colab every notebook runs on its own temporary machine, so a file saved
# here is not there when the next notebook opens. This cell keeps the
# results in your Google Drive instead: approve the access request when it
# appears. If you decline it, or have no Google Drive, the results are
# downloaded to your computer when saved and the notebook that needs them
# asks for them back. Locally this cell does nothing.
import course_core as cc
cc.keep_outputs("Ex10.1_outputs")


In [ ]:
# --- setup: every Part 2 notebook opens with this cell ------------------
# Needs course_core.py, pinn_core.py and problem.py beside this notebook.
# On Colab the files cell above fetched them from the public course repository.
import os
for f in ("course_core.py", "pinn_core.py", "problem.py"):
    assert os.path.exists(f), f"{f} is missing - run the files cell above first"

from pinn_core import *                                  # noqa: F401,F403
import problem as pb
import numpy as np, torch, matplotlib.pyplot as plt

set_seed(88)
print("device:", DEVICE, " dtype:", torch.get_default_dtype())

In [ ]:
# Shared residual and loss from notebook 01 (included here so Run All works).
# TODO 1 --- the radial diffusion residual --------------------------------------------------------
# Two `...` to replace:
#   line 1  ->  d2(c, rt, 0)                                                     c_rr
#   line 2  ->  c_t - c_rr - 2.0 / torch.clamp(r, min=pb.R_CENTRE_EPS) * c_r     c_t = c_rr + (2/r) c_r
# The clamp is one way to survive the centre; sampling with r_min=pb.R_CENTRE_EPS is another.
def residual_fn(model, rt):
    c = model(rt)
    g = grad(c, rt)
    c_r, c_t = g[:, 0:1], g[:, 1:2]
    r = rt[:, 0:1]
    c_rr = d2(c, rt, 0)
    return c_t - c_rr - 2.0 / torch.clamp(r, min=pb.R_CENTRE_EPS) * c_r
# ------------------------------------------------------------------------------

# TODO 2 --- the loss: physics, two flux conditions, initial state ------------------------------------
# Three `...` to replace:
#   line 1  ->  mse(grad(model(r_s), r_s)[:, 0:1] - 1.0)        surface flux c_r = 1 (the current arriving)
#   line 2  ->  mse(grad(model(r_0), r_0)[:, 0:1])              centre symmetry c_r = 0
#   line 3  ->  mse(model(ic))                                  initially c = 0
def loss_fn_factory(model, rt):
    n = 200
    r_s = to_tensor(pb.particle_surface_points(n, t_end=1.0), requires_grad=True)   # r = 1
    r_0 = to_tensor(pb.particle_centre_points(n, t_end=1.0), requires_grad=True)    # r = eps
    ic  = to_tensor(pb.particle_initial_points(n))                                  # t = 0, no grad needed

    def loss_fn():                                # no arguments: the contract with train_two_stage
        L_pde  = mse(residual_fn(model, rt))
        L_surf = mse(grad(model(r_s), r_s)[:, 0:1] - 1.0)
        L_cen  = mse(grad(model(r_0), r_0)[:, 0:1])
        L_ic   = mse(model(ic))
        return L_pde + 10.0 * (L_surf + L_cen + L_ic)
    return loss_fn
# ------------------------------------------------------------------------------


In [ ]:
runs = []

def on_run(cell, opts):
    r = pb.run_particle(cell, residual_fn, loss_fn_factory,
                        n_coll=opts["n_coll"], n_hidden=opts["n_hidden"],
                        n_layers=opts["n_layers"], adam=opts["adam"])
    runs.append(r)
    pb.plot_particle(r)
    q = 2.0e4 * cell.c_rate
    T = pb.lumped_temperature(cell, q, np.linspace(0, cell.t_discharge, 200))
    print(f"  estimated temperature rise: {T[-1]-cell.T_amb:.1f} K")
    pb.plot_rz_field(cell, float(T[-1] - cell.T_amb))
    print(f"\n{len(runs)} run(s) recorded")

panel = pb.control_panel(on_run)

## Suggested studies

| Study | Vary | Look for |
|---|---|---|
| Rate | C-rate 0.2 → 5 | when the particle time constant approaches the discharge duration |
| Cooling | h over three decades | where the Biot number crosses 0.1 and a lumped model stops being honest |
| Ambient | −10 → 45 °C | how much the Arrhenius dependence changes the transport |
| Sampling | N_f, network size | where the error plateaus |

In [ ]:
import pickle
os.makedirs(cc.OUTPUT_DIR, exist_ok=True)
path = os.path.join(cc.OUTPUT_DIR, "ex101_runs.pkl")
with open(path, "wb") as f:
    pickle.dump([{k: v for k, v in r.items() if k != "model"} for r in runs], f)
print(f"saved {len(runs)} runs to {path}")
cc.saved(path)


---

Continue with **[`Ex10.1_05_report.ipynb`](https://colab.research.google.com/github/Noma99-ai/Excercise_DLL_Course/blob/main/exercises/Ex10.1-spm-and-thermal/Ex10.1_05_report.ipynb)**.
